# GraphST 教程（Stereo-seq）


## 分析目标
1. 读取小鼠胚胎 E9.5 的 Stereo-seq 数据；
2. 使用 GraphST 学习空间表达表示；
3. 基于表示做空间聚类（22类）；
4. 可视化空间结构域并保存结果。

## 输入与输出
- 输入文件：`/data/work/E9.5_E1S1.MOSTA.h5ad`
- 输出文件：
  - `/data/work/Mouse_Embryo_E9.5_GraphST_clustered.h5ad`
  - `/data/work/Mouse_Embryo_E9.5_domain_labels.csv`

## 导入依赖并配置运行参数

**目的**：
- 导入 GraphST 分析所需的 Python 包；
- 定义工作目录、输入文件路径、聚类数等关键参数；
- 选择计算设备（GPU/CPU）；
- 预留 mclust 所需的 R 环境变量配置入口。

**作用**：
- 为后续读取数据、模型训练、聚类和可视化提供统一参数来源；
- 确保脚本在不同机器上可复现、可迁移。

In [1]:
import os
import warnings
import torch
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

from GraphST import GraphST
from GraphST.utils import clustering

# 忽略非关键warning，减少输出干扰（调试阶段可关闭）
warnings.filterwarnings("ignore")

# =============================
# 基础参数
# =============================
dataset = "Mouse_Embryo"  # 数据集名称（仅用于标识）
work_dir = "/data/work"  # 工作路径（用户指定）
data_file = os.path.join(work_dir, "E9.5_E1S1.MOSTA.h5ad")  # 输入h5ad文件

# 目标空间结构域数量（教程指定22）
n_clusters = 22

# 设备选择：若检测到CUDA则用GPU，否则回退CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# mclust依赖R环境：请按本机实际R路径设置
# 示例：
# os.environ["R_HOME"] = "/usr/lib/R"

print("R_HOME =", os.environ.get("R_HOME", "Not set"))
print("Input file =", data_file)

Using device: cpu
R_HOME = Not set
Input file = /data/work/E9.5_E1S1.MOSTA.h5ad


## 读取并检查输入数据

**目的**：
- 验证输入文件是否存在；
- 读取 AnnData 格式的 Stereo-seq 数据；
- 做基础结构检查（obs/obsm）。

**作用**：
- 避免路径错误导致后续计算中断；
- 明确数据对象中有哪些元信息和空间坐标字段，便于后续分析。

In [2]:
# 读取数据前先检查文件存在性，防止直接报底层IO错误
if not os.path.exists(data_file):
    raise FileNotFoundError(
        f"未找到输入文件: {data_file}\n"
        "请先下载 E9.5_E1S1.MOSTA.h5ad 并放到 /data/work 下。"
    )

# 读取h5ad为AnnData对象
adata = sc.read_h5ad(data_file)

# 保证基因名唯一，避免下游按var_names索引时冲突
adata.var_names_make_unique()

# 打印数据概览
print(adata)
print("obs columns:", adata.obs.columns.tolist()[:20])
print("obsm keys:", list(adata.obsm.keys()))

AnnData object with n_obs × n_vars = 5913 × 25568
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'annotation', 'Regulon - 2310011J03Rik', 'Regulon - 5730507C01Rik', 'Regulon - Alx1', 'Regulon - Alx3', 'Regulon - Alx4', 'Regulon - Ar', 'Regulon - Arid3a', 'Regulon - Arid3c', 'Regulon - Arnt2', 'Regulon - Arx', 'Regulon - Ascl1', 'Regulon - Atf1', 'Regulon - Atf4', 'Regulon - Atf5', 'Regulon - Atf6', 'Regulon - Atf7', 'Regulon - Bach1', 'Regulon - Bach2', 'Regulon - Barhl1', 'Regulon - Barx1', 'Regulon - Batf', 'Regulon - Bcl11a', 'Regulon - Bcl3', 'Regulon - Bcl6', 'Regulon - Bcl6b', 'Regulon - Bclaf1', 'Regulon - Bdp1', 'Regulon - Bhlha15', 'Regulon - Bhlhe22', 'Regulon - Bhlhe23', 'Regulon - Bhlhe41', 'Regulon - Bmyc', 'Regulon - Boll', 'Regulon - Bptf', 'Regulon - Brca1', 'Regulon - Brf1', 'Regulon - Brf2', 'Regulon - Bsx', 'Regulon - Cdx1', 'Regulon - Cdx2', 'Regulon - Cebpa', 'Regulon - Cebpz', 'Regulon - Chd1', 'Regulon - Clock', 'Re

## 训练 GraphST 模型

**目的**：
- 构建 GraphST 模型对象；
- 指定数据类型为 Stereo（Stereo-seq）；
- 启动训练获得空间表示。

**作用**：
- GraphST 会将空间邻域关系与基因表达信息联合建模；
- 训练后生成的嵌入表示用于后续更稳定的空间聚类。

In [ ]:
# 构建GraphST模型
# datatype='Stereo' 对应Stereo-seq数据
model = GraphST.GraphST(adata, datatype="Stereo", device=device)

# 训练模型（包含图构建与表示学习）
adata = model.train()

print("GraphST training done.")
print("obsm keys after training:", list(adata.obsm.keys()))

Graph constructed!
Building sparse matrix ...
Begin to train ST data...


 28%|██▊       | 165/600 [00:29<01:22,  5.26it/s]

## 空间聚类（domain识别）

**目的**：
- 对 GraphST 学到的低维表示进行聚类；
- 识别空间结构域（domain）。

**作用**：
- 将每个 spot/bin 分配到一个空间域标签；
- 便于后续空间分区解释、组织区域对照与生物学注释。

**说明**：
- 教程推荐 `mclust`（通常在空间数据上表现较稳）；
- `leiden/louvain` 也可用，但需要分辨率搜索参数。

In [ ]:
# 选择聚类工具：mclust / leiden / louvain
tool = "mclust"

if tool == "mclust":
    # 直接按指定簇数聚类
    clustering(adata, n_clusters, method=tool)
elif tool in ["leiden", "louvain"]:
    # leiden/louvain 常通过分辨率范围搜索近似得到目标簇数
    clustering(adata, n_clusters, method=tool, start=0.1, end=2.0, increment=0.01)
else:
    raise ValueError("tool 必须是 'mclust' / 'leiden' / 'louvain'")

print("Clustering done.")
print("domain value counts:")
print(adata.obs["domain"].value_counts().sort_index())

## 空间可视化

**目的**：
- 在空间坐标上展示 domain 聚类结果。

**作用**：
- 直观评估聚类是否形成合理的组织空间分区；
- 便于与胚胎解剖结构或已知标记区域对照。

**说明**：
- 某些数据坐标系方向与习惯图像坐标相反，可按需翻转 Y 轴。

In [ ]:
# 若显示方向与预期不一致，可启用Y轴翻转
adata.obsm["spatial"][:, 1] = -1 * adata.obsm["spatial"][:, 1]

plt.rcParams["figure.figsize"] = (5, 6)

# 22类颜色（与n_clusters一致）
plot_color = [
    "#F56867", "#556B2F", "#C798EE", "#59BE86", "#006400", "#8470FF",
    "#CD69C9", "#EE7621", "#B22222", "#FFD700", "#CD5555", "#DB4C6C",
    "#8B658B", "#1E90FF", "#AF5F3C", "#CAFF70", "#F9BD3F", "#DAB370",
    "#877F6C", "#268785", "#82EF2D", "#B4EEB4"
]

# 在spatial坐标上按domain着色
ax = sc.pl.embedding(
    adata,
    basis="spatial",
    color="domain",
    s=20,
    show=False,
    palette=plot_color,
    title="GraphST"
)

# 关闭坐标轴，突出组织轮廓与分区
ax.axis("off")
ax.set_title("Mouse Embryo E9.5")
plt.show()

## 保存分析结果

**目的**：
- 保存带有 GraphST 表示和 domain 标签的完整 AnnData；
- 导出仅含 domain 的标签表便于下游统计作图。

**作用**：
- 保证结果可复用、可追溯；
- 便于后续做 marker 分析、区域比较、跨样本对照。

In [6]:
out_h5ad = os.path.join(work_dir, "Mouse_Embryo_E9.5_GraphST_clustered.h5ad")
out_csv = os.path.join(work_dir, "Mouse_Embryo_E9.5_domain_labels.csv")

# 保存完整对象（推荐，便于复现与二次分析）
adata.write_h5ad(out_h5ad)

# 仅导出domain列（轻量，便于外部软件读取）
adata.obs[["domain"]].to_csv(out_csv)

print("Saved:")
print(" -", out_h5ad)
print(" -", out_csv)

Saved:
 - /data/work/Mouse_Embryo_E9.5_GraphST_clustered.h5ad
 - /data/work/Mouse_Embryo_E9.5_domain_labels.csv
